In [9]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_pinecone import PineconeVectorStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from pinecone import Pinecone, ServerlessSpec

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")

# -------------------------------
# STEP 1 — Load Document
# -------------------------------
print("📄 Loading document...")
loader = PyPDFLoader("sample.pdf")  # make sure file exists
documents = loader.load()
print(f"✅ Loaded {len(documents)} pages")

# -------------------------------
# STEP 2 — Split
# -------------------------------
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
chunks = splitter.split_documents(documents)
print(f"✅ Created {len(chunks)} chunks")

# -------------------------------
# STEP 3 — Embeddings
# -------------------------------
print("🔢 Loading embeddings...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
print("✅ Embeddings ready")

# -------------------------------
# STEP 4 — Pinecone (v8)
# -------------------------------
print("🌲 Setting up Pinecone...")

pc = Pinecone(api_key=PINECONE_API_KEY)

INDEX_NAME = "qa-system"

# Get existing indexes
existing_indexes = [i["name"] for i in pc.list_indexes()]

# Create index if not exists
if INDEX_NAME not in existing_indexes:
    pc.create_index(
        name=INDEX_NAME,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )
    print("✅ Index created")
else:
    print("✅ Index already exists")

# Connect to index
index = pc.Index(INDEX_NAME)

# -------------------------------
# STEP 5 — Store Embeddings
# -------------------------------
print("📤 Storing embeddings...")

vectorstore = PineconeVectorStore(
    index=index,
    embedding=embeddings
)

vectorstore.add_documents(chunks)

print("✅ Stored in Pinecone")

# -------------------------------
# STEP 6 — Retriever
# -------------------------------
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

# -------------------------------
# STEP 7 — LLM
# -------------------------------
llm = ChatGroq(
    api_key=GROQ_API_KEY,
    model_name="llama-3.3-70b-versatile",
    temperature=0
)

# -------------------------------
# STEP 8 — QA Chain
# -------------------------------
prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant. Answer using ONLY the context.

Context:
{context}

Question: {question}

Answer:
""")

def format_docs(docs):
    return "\n\n".join([d.page_content for d in docs])

qa_chain = (
    {
        "context": lambda x: format_docs(retriever.invoke(x["question"])),
        "question": lambda x: x["question"]
    }
    | prompt
    | llm
    | StrOutputParser()
)

print("✅ QA system ready!")

# -------------------------------
# STEP 9 — Test
# -------------------------------
import sys

while True:
    user_input = input("Input Prompt: ")

    if user_input.lower() == "exit":
        print("Exiting")
        sys.exit()

    if user_input.strip() == "":
        continue

    result = qa_chain.invoke({"question": user_input})

    print("Answer:", result)

📄 Loading document...
✅ Loaded 113 pages
✅ Created 419 chunks
🔢 Loading embeddings...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Embeddings ready
🌲 Setting up Pinecone...
✅ Index already exists
📤 Storing embeddings...
✅ Stored in Pinecone
✅ QA system ready!


Input Prompt:  can you summarize pdf?


...

Answer: Yes.


Input Prompt:  exit


Exiting


SystemExit: 

C:\Users\hp\anaconda3\envs\ml_env\lib\site-packages\IPython\core\interactiveshell.py:3587: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [7]:
question = "Explain Naïve Bayes Classifier with an Example"
answer = qa_chain.invoke({"question": question})
print(answer)

Naive Bayes classifiers are a collection of classification algorithms based on Bayes' Theorem. They are not a single algorithm but a family of algorithms where all of them share a common principle, i.e., every pair of features being classified is independent of each other. 

The assumption is that feature attributes are independent of each other. 

For example, consider the given dataset: 
Sl. No. Color Legs Height Smelly Species 
1 White 3 Short Yes M 
2 Green 2 Tall No M 
3 Green 3 Short Yes M 
4 White 3 Short Yes M 
5 Green 2 Short No H 
6 White 2 Tall No H 
7 White 2 Tall No H 
8 White 2 Short Yes H 

To construct a Naive Bayes Classifier, we would use this dataset to calculate probabilities and classify new instances based on these probabilities, assuming that the features (Color, Legs, Height, Smelly) are independent of each other.
